# Assignment 2 Step 1 — measured CPU baseline

左承臻 113700058。這份 notebook 會建立完整專案、下載老師資料、跑五種比較、檢查時間洩漏與官方分數、產生 Q1 DOT 圖。

**Kaggle：Import Notebook → Internet 開啟 → Accelerator 選 None → Save & Run All。Colab 也可直接跑。這一步不需要 GPU。**

我已在 10/22 資料快照實際跑完第一輪；你不必為了交回相同結果立刻重跑。此 notebook 是讓你保留並重現程式。老師資料每天覆寫，新快照的結果可能改變。

資料預測期間是 11/12–11/19；官方測試名單 10/28 才公布，這份不是最終提交檔。歷史驗證分數不是正式分數。現在沒有實作 GNN，也沒有宣稱新增關聯特徵已超越時間模型。


In [ ]:
import json, sys, subprocess, importlib.util
from pathlib import Path
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
PROJECT = BASE / 'Asg2_Project'
PROJECT.mkdir(parents=True, exist_ok=True)
SOURCE_FILES = json.loads(r'''{"requirements.txt": "numpy\npandas\npyarrow\nlightgbm\nscikit-learn\ngraphviz\n", ".gitignore": "__pycache__/\n.ipynb_checkpoints/\n.venv/\nartifacts/cache/\n*.pkl\nassignment2_data/\n.env\n", "README.md": "# NYCU Big Data Assignment 2 — first measured baseline\n\nAuthor: Chen Jhen Tso (113700058). Individual assignment, Fall 2026.\n\nThis is a functioning CPU baseline and experiment scaffold, **not a completed November submission or a claim of full marks**. It uses only the instructor's official relational data. No GitHub user API or external activity data are collected.\n\n## Quick start\n\nThe accompanying `Asg2_Start.ipynb` is self-contained: import into Kaggle/Colab, keep Internet on, and Run All. GPU is not needed. It writes this project outside the downloaded instructor-data folder and runs the benchmark, checks and graph extraction. First-run package installation/download and machine speed determine runtime; no specific runtime is guaranteed.\n\nFor local execution (from this folder):\n\n```powershell\npython -m pip install -r requirements.txt\ngit clone --depth 1 https://github.com/nycubigdata-2026fall/BigData2026-Assignment2.git ../assignment2_data\npython -m asg2.run benchmark --data ../assignment2_data --out artifacts\npython scripts/checks.py --data ../assignment2_data --out artifacts\npython scripts/graphs.py --data ../assignment2_data --out artifacts/graphs\n```\n\nGraph extraction always writes DOT. SVG rendering requires Graphviz's `dot` executable (the Python package alone does not supply it). Measured SVG/PNG diagrams are already included under `artifacts/graphs/`. Local measurement used a Graphviz WebAssembly renderer because system `dot` was unavailable; the DOT input is the same.\n\n## What has actually run\n\nThe baseline was run on the instructor snapshot whose latest observed row timestamp is **2026-10-22 11:59:23 UTC**. `artifacts/data_manifest.json` records SHA256 for all eight files. Every parquet is read with a column allowlist. The six known-only-later fields are excluded from modeling entirely.\n\nTraining origins are Thursdays in Taiwan from May 7 to September 10, 2026 (19 weeks, 148,917 user-week samples). Calibration is September 24 (4,931 samples). Two separate historical holdout origins are October 8 and October 15 (8,731 samples total). All labels are seven-day windows. Calibration thresholds and model selection use ONLY calibration outcomes, never holdout results. A four-hour observation gap is simulated at every origin to match the final release ending Nov11 20:00 Taiwan before the Nov12 00:00 forecast start. Prior outcome windows end before the next split's observation cutoff.\n\nHistorical population is a proxy: cohort users with at least five observed target events in the previous 90 days. This does not claim to reconstruct the exact official Oct28 test cohort selection. Final prediction uses the official test IDs directly with no activity-based filtering.\n\n| Model | Macro F1 | Micro F1 | Official combined score |\n|---|---:|---:|---:|\n| Last-week rule | 40.39% | 52.26% | 46.32% |\n| Lifetime user features | 36.84% | 48.61% | 42.73% |\n| User temporal features | 44.55% | 55.79% | 50.17% |\n| Static relational features | 37.33% | 49.53% | 43.43% |\n| Temporal plus relational features | 44.14% | 55.44% | 49.79% |\n\nThese are **historical validation results, not the official November score**. Temporal-only was selected by calibration. Current relational features did not improve the held-out combined score over temporal-only; this result is retained rather than hidden. Further experiments need a new untouched forward holdout as more data arrive; do not repeatedly tune to October 8/15.\n\n## Models and features\n\nSix LightGBM binary classifiers, one per target. Fixed seed 42, 160 trees, learning rate .045, 23 leaves, minimum leaf support 70, L2 4. The variants are:\n\n- `static`: user age and six lifetime activity counts.\n- `temporal`: adds 7/14/30/90-day counts, active days, distinct repos, last-event recency and recent/longer-term rate ratios.\n- `relational`: static user features plus user-repo-user neighborhood aggregation and cross-table PR/issue-author interaction counts.\n- `full`: all above plus 30-day relational aggregates.\n\nRelations use FKs to connect activity to users/repositories and join review/discussion events to PR/issue authors. Repository activity excludes each user's own contribution when constructing neighboring activity. Non-cohort activity is incomplete by construction; neighborhood statistics describe the supplied observed graph, not complete GitHub traffic. This is relational feature aggregation followed by tree models, **not a GNN or end-to-end RDL architecture**. Whether a final submission based only on aggregated features meets the instructor's phrase “directly analyze relational databases” should be confirmed; adding an explicit relational graph model is a planned comparison, not yet implemented.\n\nThresholds are calibrated using the exact `(macro_F1 + micro_F1)/2` objective, including the official convention that an empty target with no false predictions has F1=1. Each threshold search is confined to calibration data. A simple last-week rule is included to guard against mistaking model complexity for improvement.\n\n## Time and identity correctness\n\n- Filter PR/issue/review rows by `first_seen_at`, and event occurrences by their own timestamps, before each prediction cutoff. A historical event learned only after cutoff cannot enter features.\n- Users/repositories must also exist before cutoff. Training populations use cohort users only.\n- Training labels must have occurrence within the outcome week AND availability before that week ends; retrospective records learned later are not used to train a historical model. Historical labels remain reconstructions from a released snapshot; there is no claim of access to the TAs' private truth.\n- Output uses `Users.github_user_id`; table row-index PK/FK values are used only for joins within one snapshot.\n- No random train/validation split. Date windows, row counts, data hashes, per-target scores, thresholds and checks are saved.\n- Cache directories are keyed by data hashes. After changing feature code, remove `artifacts/cache` before rerunning, or use a new output folder.\n\n`checks.py` has been executed: injecting future-availability records or future-occurrence records did not change past features; output IDs are stable GitHub IDs; scoring matches the official evaluator including empty-class behavior; split windows are correctly separated.\n\n## Q1 diagrams\n\n`artifacts/graphs/schema.svg`: all eight tables, PK/FK ports and 20 FK edges, including the self-reference in PRReviewComments. Org IDs and head-repository GitHub IDs are not row-index FKs according to the instructor schema.\n\n`entity_graph.svg`: seed GitHub IDs 22996444 and 28983449, activity-row times in `[2026-09-15 00:00, 2026-09-17 00:00)` UTC. Blue rows are activity rows in the window; yellow nodes are seed users. Gray nodes are necessary FK endpoints that may predate the window, recursively included as context. No future rows or unrelated repo events are added. Current snapshot gives 30 nodes, 57 edges, 17 in-window activity rows. Internal row IDs may differ in subsequent releases. This is an explicit extraction convention; verify the TA's expected neighborhood definition before final slides.\n\n## Final submission is not generated yet\n\nOfficial `test_input.csv` is scheduled for Oct28. The final data release is Nov11 at 21:00 Taiwan (through 20:00); deadline is Nov11 23:55. Never submit the historical holdout CSVs as final predictions.\n\nAfter final model/threshold choice is frozen and the last data release is available:\n\n```powershell\npython -m asg2.run predict --data ../assignment2_data --out artifacts --input ../assignment2_data/evaluation/test_input.csv --student-id 113700058\n```\n\nThis refits the calibration-selected variant using complete historical windows from the final snapshot, keeps calibrated thresholds, writes `artifacts/output_113700058.csv`, and invokes the official evaluator with the exact test-user list. The final refit path is implemented but cannot be executed/validated against future November data today. The program refuses an October snapshot for official final prediction. Final model rankings/thresholds may change after justified development experiments; document any change and evaluate on a fresh forward holdout.\n\nYou still need a public GitHub repository, English slides using the teacher's template (<=10 pages, Q1-Q4), and a personally narrated 8-12 minute MP4. No AI-generated presentation video. Model/slide claims must match actual implementation and saved outputs.\n\n## Source attribution\n\nData, schema and evaluator: https://github.com/nycubigdata-2026fall/BigData2026-Assignment2 . Model and experiment code in this package was prepared with AI assistance. No results are fabricated. Dependencies in `requirements.txt` are direct requirements; actual local versions are saved in `artifacts/environment.json`.\n", "asg2/run.py": "\"\"\"Train, calibrate and hold out time-separated weeks. Never use holdout to tune thresholds.\"\"\"\nimport argparse,csv,json,pickle,subprocess,time,hashlib\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom lightgbm import LGBMClassifier\nfrom .core import *\n\nclass Constant:\n    def __init__(self,p):self.p=p\n    def predict_proba(self,X):return np.tile([1-self.p,self.p],(len(X),1))\n\ndef fit_models(X,y,columns,trees=160):\n    models=[]\n    for j,kind in enumerate(TARGETS):\n        if len(np.unique(y[:,j]))<2:m=Constant(float(y[:,j].mean()))\n        else:\n            m=LGBMClassifier(n_estimators=trees,learning_rate=.045,num_leaves=23,max_depth=-1,min_child_samples=70,reg_lambda=4,subsample=.85,subsample_freq=1,colsample_bytree=.85,random_state=42,n_jobs=4,verbosity=-1)\n            m.fit(X[columns],y[:,j])\n        models.append(m)\n    return models\n\ndef predict(models,X,columns):return np.column_stack([m.predict_proba(X[columns])[:,1] for m in models])\n\ndef fingerprint(data):\n    base=Path(data)/'data/db'\n    if not base.exists():base=Path(data)\n    out={}\n    for name in TABLES:\n        p=base/f'{name}.parquet';h=hashlib.sha256()\n        with p.open('rb') as f:\n            for block in iter(lambda:f.read(1024*1024),b''):h.update(block)\n        out[name]={'sha256':h.hexdigest(),'bytes':p.stat().st_size}\n    return out\n\ndef make_samples(tables,events,origins,cache,gap):\n    frames=[];ys=[]\n    coverage=events.available_at.max()\n    for t in origins:\n        t=stamp(t)\n        if t+pd.Timedelta(days=7)>coverage:raise ValueError(f'Outcome week not yet fully observed: {t}')\n        key=t.strftime('%Y%m%dT%H%M')+f'_gap{gap}'\n        path=cache/f'{key}.pkl'\n        if path.exists():\n            with path.open('rb') as f:X,y=pickle.load(f)\n            print('Cached',key,len(X),flush=True)\n        else:\n            X=features(tables,events,t,gap_hours=gap);y=labels(tables,events,t,X.UserID)\n            with path.open('wb') as f:pickle.dump((X,y),f)\n            print('Built',key,len(X),flush=True)\n        if not len(X):raise ValueError('No eligible users')\n        frames.append(X);ys.append(y)\n    return pd.concat(frames,ignore_index=True),np.concatenate(ys)\n\ndef dates(start,end):return pd.date_range(start,end,freq='7D',tz='Asia/Taipei').tz_convert('UTC').tz_localize(None)\n\ndef benchmark(args):\n    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)\n    tables=load_tables(args.data);events=build_events(tables)\n    manifest=fingerprint(args.data)\n    digest=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()[:16]\n    cache=out/'cache'/digest;cache.mkdir(parents=True,exist_ok=True)\n    (out/'data_manifest.json').write_text(json.dumps(manifest,indent=2))\n    train_dates=dates(args.train_start,'2026-09-17');cal_dates=dates('2026-09-24','2026-10-01');test_dates=dates('2026-10-08','2026-10-15')\n    # Outcomes of every preceding split finish before the next split's feature cutoff.\n    # Last training origin Sep17 has outcomes through Sep24; calibration's earliest\n    # feature cutoff is Sep23 20:00, so exclude Sep17 to preserve the 4-hour gap.\n    # Likewise keep a full calendar week between calibration and holdout origins.\n    train_dates=train_dates[train_dates+pd.Timedelta(days=7)<=cal_dates.min()-pd.Timedelta(hours=args.gap_hours)]\n    cal_dates=cal_dates[cal_dates+pd.Timedelta(days=7)<=test_dates.min()-pd.Timedelta(hours=args.gap_hours)]\n    Xtr,ytr=make_samples(tables,events,train_dates,cache,args.gap_hours)\n    Xcal,ycal=make_samples(tables,events,cal_dates,cache,args.gap_hours)\n    Xtest,ytest=make_samples(tables,events,test_dates,cache,args.gap_hours)\n    results=[]\n    # A rule baseline uses observed activity in the last 7 days, no fitting.\n    def lastweek(X):return np.column_stack([X[f't_count7_{k}'].to_numpy()>0 for k in TARGETS])\n    rule=score(ytest,lastweek(Xtest));rule.update(model='last_week_rule',split='holdout');results.append(rule)\n    best=None;best_cal=-1\n    probabilities={}\n    for variant in ['static','temporal','relational','full']:\n        cols=feature_columns(Xtr,variant);t0=time.perf_counter()\n        models=fit_models(Xtr,ytr,cols,args.trees)\n        probcal=predict(models,Xcal,cols)\n        thresholds=tune_thresholds(ycal,probcal)\n        probtest=predict(models,Xtest,cols);probabilities[variant]=probtest\n        calscore=score(ycal,probcal>=thresholds)\n        if calscore['score']>best_cal:best_cal=calscore['score'];best=variant\n        row=score(ytest,probtest>=thresholds)\n        row.update(model=variant,split='holdout',n_features=len(cols),fit_seconds=time.perf_counter()-t0,calibration_score=calscore['score'])\n        results.append(row)\n        for origin in sorted(Xtest.origin.unique()):\n            mask=Xtest.origin.eq(origin).to_numpy()\n            results.append({**score(ytest[mask],probtest[mask]>=thresholds),'model':variant,'split':str(origin),'n_users':int(mask.sum())})\n        with (out/f'model_{variant}.pkl').open('wb') as f:pickle.dump({'models':models,'columns':cols,'thresholds':thresholds,'variant':variant},f)\n        (out/f'thresholds_{variant}.json').write_text(json.dumps(dict(zip(TARGETS,map(float,thresholds))),indent=2))\n        pred=pd.DataFrame((probtest>=thresholds).astype(int),columns=TARGETS);pred.insert(0,'UserID',Xtest.UserID.to_numpy());pred.insert(1,'origin',Xtest.origin)\n        pred.to_csv(out/f'holdout_predictions_{variant}.csv',index=False)\n        print('RESULT',row,flush=True)\n    df=pd.DataFrame(results);df.to_csv(out/'metrics.csv',index=False)\n    protocol={'train_origins':[str(x) for x in train_dates],'calibration_origins':[str(x) for x in cal_dates],'holdout_origins':[str(x) for x in test_dates],'train_rows':len(Xtr),'calibration_rows':len(Xcal),'holdout_rows':len(Xtest),'observation_gap_hours':args.gap_hours,'data_latest_available_timestamp':str(events.available_at.max()),'selected_by_calibration_only':best,'selected_calibration_score':best_cal,'seed':42,'threshold_objective':'(macro F1 + micro F1)/2, official empty-class convention','historical_eligibility':'cohort users with >=5 observed target events in past90 days (proxy for test selection)','warning':'Temporal holdout is historical, not official November test performance.'}\n    (out/'protocol.json').write_text(json.dumps(protocol,indent=2))\n    print('\\nComplete. Selected using calibration ONLY:',best)\n    print(df[df.split=='holdout'].to_string(index=False))\n\ndef final_predict(args):\n    out=Path(args.out);protocol=json.loads((out/'protocol.json').read_text());variant=protocol['selected_by_calibration_only']\n    with (out/f'model_{variant}.pkl').open('rb') as f:chosen=pickle.load(f)\n    tables=load_tables(args.data);events=build_events(tables)\n    ids=pd.read_csv(args.input).UserID.to_numpy()\n    target=stamp('2026-11-12 00:00:00+08:00');observation_end=target-pd.Timedelta(hours=4)\n    if events.available_at.max()<pd.Timestamp('2026-11-11 11:00:00'):\n        raise ValueError('Official final predictions require the final Nov11 release. Do not submit an October dry run.')\n    # Refitting includes only complete outcome windows before the observation cutoff.\n    start=protocol['train_origins'][0]\n    origins=pd.date_range(start,target-pd.Timedelta(days=7),freq='7D')\n    origins=origins[origins+pd.Timedelta(days=7)<=observation_end]\n    manifest=fingerprint(args.data);digest=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()[:16]\n    cache=out/'cache'/digest;cache.mkdir(parents=True,exist_ok=True)\n    X,y=make_samples(tables,events,origins,cache,4)\n    cols=chosen['columns'];models=fit_models(X,y,cols,args.trees)\n    Xtest=features(tables,events,target,user_ids=ids,gap_hours=4)\n    pred=predict(models,Xtest,cols)>=chosen['thresholds']\n    df=pd.DataFrame(pred.astype('int8'),columns=TARGETS);df.insert(0,'UserID',Xtest.UserID.to_numpy())\n    destination=out/f'output_{args.student_id}.csv';df.to_csv(destination,index=False)\n    # Run the OFFICIAL evaluator, which checks both header and exact user set.\n    cmd=['python',str(Path(args.data)/'evaluation/evaluate.py'),'--pred',str(destination),'--input',args.input]\n    subprocess.run(cmd,check=True)\n    (out/'final_protocol.json').write_text(json.dumps({'model':variant,'target_start_utc':str(target),'feature_cutoff_utc':str(observation_end),'refit_origins':[str(t) for t in origins],'thresholds_from_historical_calibration':chosen['thresholds'].tolist(),'data_manifest':manifest},indent=2))\n    print('Official-format prediction file:',destination)\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument('action',choices=['benchmark','predict']);p.add_argument('--data',required=True);p.add_argument('--out',default='artifacts');p.add_argument('--train-start',default='2026-05-07');p.add_argument('--trees',type=int,default=160);p.add_argument('--gap-hours',type=int,default=4);p.add_argument('--input');p.add_argument('--student-id',default='113700058')\n    a=p.parse_args()\n    if a.action=='predict' and not a.input:p.error('--input is required for predict')\n    benchmark(a) if a.action=='benchmark' else final_predict(a)\n\nif __name__=='__main__':main()\n", "asg2/core.py": "\"\"\"As-of relational features using only the official eight-table release.\"\"\"\nfrom pathlib import Path\nimport hashlib,json\nimport numpy as np\nimport pandas as pd\n\nTARGETS=['open_pr','open_issue','issue_comment','pr_review','pr_review_comment','push']\nTABLES=['Users','Repositories','PullRequests','Issues','IssueComments','PRReviews','PRReviewComments','Pushes']\nTIME={'Users':'first_seen_at','Repositories':'first_seen_at','PullRequests':'first_seen_at','Issues':'first_seen_at','PRReviews':'first_seen_at','IssueComments':'created_at','PRReviewComments':'created_at','Pushes':'created_at'}\n# Known-only-later columns are deliberately excluded entirely, rather than guessed.\nCOLS={'Users':['user_id','github_user_id','is_cohort','first_seen_at'],\n'Repositories':['repo_id','first_seen_at'],\n'PullRequests':['pr_id','repo_id','author_id','created_at','first_seen_at'],\n'Issues':['issue_id','repo_id','author_id','assignee_id','pull_request_id','is_pull_request','created_at','first_seen_at'],\n'IssueComments':['comment_id','issue_id','repo_id','author_id','created_at'],\n'PRReviews':['review_id','pr_id','repo_id','reviewer_id','submitted_at','first_seen_at'],\n'PRReviewComments':['comment_id','pr_id','repo_id','author_id','pull_request_review_id','in_reply_to_id','created_at'],\n'Pushes':['push_id','repo_id','user_id','created_at']}\nPK={'Users':'user_id','Repositories':'repo_id','PullRequests':'pr_id','Issues':'issue_id','IssueComments':'comment_id','PRReviews':'review_id','PRReviewComments':'comment_id','Pushes':'push_id'}\nFKS={'Users':{},'Repositories':{},'PullRequests':{'repo_id':'Repositories','author_id':'Users','merged_by_id':'Users'},'Issues':{'repo_id':'Repositories','author_id':'Users','assignee_id':'Users','pull_request_id':'PullRequests'},'IssueComments':{'issue_id':'Issues','repo_id':'Repositories','author_id':'Users'},'PRReviews':{'pr_id':'PullRequests','repo_id':'Repositories','reviewer_id':'Users'},'PRReviewComments':{'pr_id':'PullRequests','repo_id':'Repositories','author_id':'Users','pull_request_review_id':'PRReviews','in_reply_to_id':'PRReviewComments'},'Pushes':{'user_id':'Users','repo_id':'Repositories'}}\n\ndef stamp(x):\n    t=pd.Timestamp(x)\n    return t.tz_convert('UTC').tz_localize(None) if t.tzinfo is not None else t\n\ndef load_tables(data):\n    base=Path(data)/'data/db'\n    if not base.exists():base=Path(data)\n    tables={t:pd.read_parquet(base/f'{t}.parquet',columns=COLS[t]) for t in TABLES}\n    for t,d in tables.items():\n        for c in d:\n            if c.endswith('_at'):d[c]=pd.to_datetime(d[c])\n        if d[PK[t]].duplicated().any():raise ValueError(f'Duplicate PK: {t}')\n    return tables\n\ndef build_events(tables):\n    specs=[('PullRequests','author_id','created_at','open_pr'),('Issues','author_id','created_at','open_issue'),('IssueComments','author_id','created_at','issue_comment'),('PRReviews','reviewer_id','submitted_at','pr_review'),('PRReviewComments','author_id','created_at','pr_review_comment'),('Pushes','user_id','created_at','push')]\n    frames=[]\n    for name,actor,event_time,kind in specs:\n        d=tables[name]\n        if name=='Issues':d=d[~d.is_pull_request.fillna(False)]\n        f=pd.DataFrame({'user_id':d[actor],'repo_id':d.repo_id,'event_time':d[event_time],'available_at':d[TIME[name]],'kind':kind})\n        f=f.dropna(subset=['user_id','repo_id','event_time','available_at'])\n        f[['user_id','repo_id']]=f[['user_id','repo_id']].astype('int64')\n        frames.append(f)\n    events=pd.concat(frames,ignore_index=True)\n    # Any row with availability later than occurrence must satisfy BOTH cutoffs.\n    return events.sort_values('event_time',kind='stable').reset_index(drop=True)\n\ndef asof_events(events,t):\n    return events[(events.event_time<t)&(events.available_at<t)]\n\ndef counts(events,index):\n    return events.groupby(['user_id','kind']).size().unstack(fill_value=0).reindex(index=index,columns=TARGETS,fill_value=0).fillna(0)\n\ndef features(tables,events,origin,user_ids=None,gap_hours=4):\n    origin=stamp(origin);t=origin-pd.Timedelta(hours=gap_hours)\n    users=tables['Users'];u=users[(users.first_seen_at<t)&users.is_cohort].set_index('user_id')\n    past=asof_events(events,t)\n    # Enforce endpoint availability as well as event-row availability.\n    valid_u=set(users.loc[users.first_seen_at<t,'user_id'])\n    valid_r=set(tables['Repositories'].loc[tables['Repositories'].first_seen_at<t,'repo_id'])\n    past=past[past.user_id.isin(valid_u)&past.repo_id.isin(valid_r)]\n    recent90=past[past.event_time>=t-pd.Timedelta(days=90)]\n    if user_ids is None:\n        active=recent90.groupby('user_id').size()\n        u=u.loc[u.index.intersection(active[active>=5].index)]\n    else:\n        ids=pd.Index(np.asarray(user_ids,dtype=np.int64))\n        lookup=users.set_index('github_user_id')\n        if not ids.is_unique:raise ValueError('Duplicate UserID in input')\n        if not ids.isin(lookup.index).all():raise ValueError('Unknown official UserID')\n        mapped=lookup.loc[ids]\n        if not mapped.is_cohort.all() or not (mapped.first_seen_at<t).all():\n            raise ValueError('Input contains non-cohort or not-yet-seen users')\n        u=mapped.reset_index().set_index('user_id')\n    idx=u.index;X=pd.DataFrame(index=idx)\n    X['u_age_days']=(t-u.first_seen_at).dt.total_seconds()/86400\n    life=counts(past,idx)\n    for kind in TARGETS:X[f'u_lifetime_{kind}']=np.log1p(life[kind])\n    windows={}\n    for days in [7,14,30,90]:\n        e=past[past.event_time>=t-pd.Timedelta(days=days)]\n        c=counts(e,idx);windows[days]=c\n        for kind in TARGETS:X[f't_count{days}_{kind}']=np.log1p(c[kind])\n        X[f't_active_days{days}']=e.assign(day=e.event_time.dt.floor('D')).groupby('user_id').day.nunique().reindex(idx,fill_value=0)\n        X[f't_repos{days}']=e.groupby('user_id').repo_id.nunique().reindex(idx,fill_value=0)\n    for kind in TARGETS:\n        last=past[past.kind==kind].groupby('user_id').event_time.max().reindex(idx)\n        X[f't_recency_{kind}']=((t-last).dt.total_seconds()/86400).fillna(400).clip(upper=400)\n        X[f't_trend_{kind}']=(windows[7][kind]+1)/(windows[30][kind]/4+1)\n    # User -> repo -> users: degree and neighborhood activity, with own counts removed.\n    for days,prefix in [(None,'r_static'),(30,'r_temporal')]:\n        e=past if days is None else past[past.event_time>=t-pd.Timedelta(days=days)]\n        edge=e.groupby(['user_id','repo_id']).size().rename('own').reset_index()\n        if edge.empty:continue\n        repo=e.groupby('repo_id').agg(total=('user_id','size'),contributors=('user_id','nunique'))\n        edge=edge.join(repo,on='repo_id');edge['other']=edge.total-edge.own\n        for field in ['other','contributors']:\n            z=edge.groupby('user_id')[field].agg(['mean','max']).reindex(idx).fillna(0)\n            for stat in ['mean','max']:X[f'{prefix}_{field}_{stat}']=np.log1p(z[stat])\n        X[f'{prefix}_degree']=np.log1p(edge.groupby('user_id').repo_id.nunique().reindex(idx,fill_value=0))\n        X[f'{prefix}_concentration']=edge.groupby('user_id').own.max().reindex(idx,fill_value=0)/edge.groupby('user_id').own.sum().reindex(idx,fill_value=1)\n        for kind in TARGETS:\n            totals=e[e.kind==kind].groupby('repo_id').size()\n            own=e[e.kind==kind].groupby(['user_id','repo_id']).size()\n            z=edge[['user_id','repo_id']].copy()\n            z['other_kind']=z.repo_id.map(totals).fillna(0)-pd.MultiIndex.from_frame(z).map(own).fillna(0)\n            X[f'{prefix}_neighbor_{kind}']=np.log1p(z.groupby('user_id').other_kind.mean().reindex(idx,fill_value=0))\n    # PR/issue author <- discussion/review actor: interaction beyond shared repos.\n    for name,foreign,parent,actor in [('IssueComments','issue_id','Issues','author_id'),('PRReviews','pr_id','PullRequests','reviewer_id'),('PRReviewComments','pr_id','PullRequests','author_id')]:\n        d=tables[name];parent_rows=tables[parent]\n        p=parent_rows[parent_rows[TIME[parent]]<t][[PK[parent],'author_id']].rename(columns={'author_id':'owner_id'})\n        d=d[d[TIME[name]]<t].merge(p,left_on=foreign,right_on=PK[parent],how='inner',validate='many_to_one')\n        d=d[d[actor]!=d.owner_id]\n        for days,prefix in [(None,'r_static'),(30,'r_temporal')]:\n            e=d if days is None else d[d[TIME[name]]>=t-pd.Timedelta(days=days)]\n            X[f'{prefix}_received_{name}']=np.log1p(e.groupby('owner_id').size().reindex(idx,fill_value=0))\n            X[f'{prefix}_partners_{name}']=np.log1p(e.groupby('owner_id')[actor].nunique().reindex(idx,fill_value=0))\n    X=X.replace([np.inf,-np.inf],np.nan).fillna(0).astype('float32')\n    X.insert(0,'UserID',u.github_user_id.astype('int64'))\n    X.insert(1,'origin',origin)\n    return X.reset_index(drop=True)\n\ndef labels(tables,events,origin,ids,horizon_days=7):\n    origin=stamp(origin);end=origin+pd.Timedelta(days=horizon_days)\n    # Historical labels must be available by the END of their outcome window.\n    e=events[(events.event_time>=origin)&(events.event_time<end)&(events.available_at<end)]\n    e=e.merge(tables['Users'][['user_id','github_user_id']],on='user_id',validate='many_to_one')\n    c=e.groupby(['github_user_id','kind']).size().unstack(fill_value=0)\n    return c.reindex(index=ids,columns=TARGETS,fill_value=0).fillna(0).gt(0).astype('int8').to_numpy()\n\ndef score(y,p):\n    y=np.asarray(y);p=np.asarray(p)\n    tp=((y==1)&(p==1)).sum(0);fp=((y==0)&(p==1)).sum(0);fn=((y==1)&(p==0)).sum(0)\n    den=2*tp+fp+fn;per=np.divide(2*tp,den,out=np.ones(6),where=den!=0)\n    total=den.sum();micro=float(2*tp.sum()/total) if total else 1.0\n    return {'macro_f1':float(per.mean()),'micro_f1':micro,'score':float((per.mean()+micro)/2),**dict(zip(TARGETS,map(float,per)))}\n\ndef tune_thresholds(y,prob):\n    thresholds=np.full(6,.5)\n    grid=np.linspace(.05,.75,29)\n    # Optimize the official combined score, rather than six F1 scores separately.\n    for _ in range(3):\n        for j in range(6):\n            candidates=[]\n            for v in grid:\n                t=thresholds.copy();t[j]=v\n                candidates.append(score(y,prob>=t)['score'])\n            thresholds[j]=grid[int(np.argmax(candidates))]\n    return thresholds\n\ndef feature_columns(frame,variant):\n    names=[c for c in frame if c not in ['UserID','origin']]\n    return [c for c in names if c.startswith('u_') or (variant in ['temporal','full'] and c.startswith('t_')) or (variant in ['relational','full'] and c.startswith('r_static_')) or (variant=='full' and c.startswith('r_temporal_'))]\n", "asg2/__init__.py": "\"\"\"NYCU Big Data Assignment 2 pipeline.\"\"\"\n", "scripts/checks.py": "\"\"\"Focused checks for leakage, split timing, identifiers and official scoring.\"\"\"\nimport sys,json,importlib.util\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport numpy as np\nimport pandas as pd\nfrom asg2.core import *\n\ndef main(data,out):\n    tables=load_tables(data);events=build_events(tables);t=stamp('2026-09-24T00:00:00+08:00')\n    x=features(tables,events,t)\n    future=events.iloc[:100].copy()\n    future['available_at']=t+pd.Timedelta(days=100)\n    future['event_time']=t-pd.Timedelta(days=1)\n    x2=features(tables,pd.concat([events,future],ignore_index=True),t)\n    pd.testing.assert_frame_equal(x,x2)\n    future['event_time']=t+pd.Timedelta(days=100)\n    future['available_at']=t-pd.Timedelta(days=1)\n    x3=features(tables,pd.concat([events,future],ignore_index=True),t)\n    pd.testing.assert_frame_equal(x,x3)\n    assert x.UserID.is_unique\n    assert set(x.UserID).issubset(set(tables['Users'].github_user_id))\n    # Official empty-class F1 convention differs from sklearn's usual default.\n    module=Path(data)/'evaluation/evaluate.py'\n    spec=importlib.util.spec_from_file_location('official_evaluation',module)\n    official=importlib.util.module_from_spec(spec);spec.loader.exec_module(official)\n    rng=np.random.default_rng(42)\n    for y,p in [(np.zeros((8,6),int),np.zeros((8,6),int)),(rng.integers(0,2,(100,6)),rng.integers(0,2,(100,6)))]:\n        truth=pd.DataFrame(y,columns=TARGETS);truth.insert(0,'UserID',np.arange(len(y)))\n        pred=pd.DataFrame(p,columns=TARGETS);pred.insert(0,'UserID',np.arange(len(y)))\n        expected=official.scores(pred,truth);got=score(y,p)\n        for k in ['macro_f1','micro_f1','score']:assert abs(got[k]-expected[k])<1e-12\n    protocol=json.loads((Path(out)/'protocol.json').read_text())\n    gap=pd.Timedelta(hours=protocol['observation_gap_hours'])\n    assert max(map(stamp,protocol['train_origins']))+pd.Timedelta(days=7)<=min(map(stamp,protocol['calibration_origins']))-gap\n    assert max(map(stamp,protocol['calibration_origins']))+pd.Timedelta(days=7)<=min(map(stamp,protocol['holdout_origins']))-gap\n    result={'future_availability_excluded':True,'future_occurrence_excluded':True,'stable_github_output_ids':True,'official_score_equivalence_including_empty_classes':True,'nonoverlapping_train_calibration_holdout':True}\n    (Path(out)/'checks.json').write_text(json.dumps(result,indent=2))\n    print('CHECKS PASSED:',result)\n\nif __name__=='__main__':\n import argparse\n p=argparse.ArgumentParser();p.add_argument('--data',required=True);p.add_argument('--out',default='artifacts');a=p.parse_args();main(a.data,a.out)\n", "scripts/graphs.py": "\"\"\"Q1: every declared FK, plus the actual two-user as-of entity graph.\"\"\"\nimport sys,json\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport pandas as pd\nfrom graphviz import Digraph\nfrom asg2.core import TABLES,PK,FKS,TIME,COLS,stamp\n\ndef main(data,out):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    g=Digraph('schema',format='svg');g.attr(rankdir='LR',fontname='Arial');g.attr('node',shape='plain',fontname='Arial',fontsize='11')\n    for name in TABLES:\n        fields=[(PK[name], f'{PK[name]} (PK)')]\n        fields += [(c, f'{c} (FK)') for c in FKS[name]]\n        if name=='Users':fields += [('stable', 'github_user_id (stable ID)')]\n        label='<<TABLE BORDER=\"0\" CELLBORDER=\"1\" CELLSPACING=\"0\" CELLPADDING=\"5\">'\n        label+=f'<TR><TD BGCOLOR=\"#d8e7f0\"><B>{name}</B></TD></TR>'\n        for port,text in fields:label+=f'<TR><TD PORT=\"{port}\" ALIGN=\"LEFT\">{text}</TD></TR>'\n        label+='</TABLE>>'\n        g.node(name,label)\n    for table,cols in FKS.items():\n        for column,target in cols.items():g.edge(table+':'+column,target+':'+PK[target],label=column,fontsize='8')\n    g.save(str(out/'schema.dot'))\n    try:g.render(str(out/'schema'),cleanup=True)\n    except Exception as e:print('DOT saved; SVG needs the Graphviz dot executable:',e)\n    base=Path(data)/'data/db'\n    ts={}\n    end=stamp('2026-09-17T00:00:00Z');start=stamp('2026-09-15T00:00:00Z')\n    for table in TABLES:\n        columns=list(COLS[table])\n        if table=='PullRequests':columns+=['merged_by_id','merged_at']\n        d=pd.read_parquet(base/f'{table}.parquet',columns=columns)\n        d=d[d[TIME[table]]<end].copy()\n        if table=='PullRequests':d.loc[d.merged_at.isna()|(d.merged_at>=end),'merged_by_id']=pd.NA\n        ts[table]=d.set_index(PK[table],drop=False)\n    users=ts['Users'];seeds=set(map(int,users.loc[users.github_user_id.isin([22996444,28983449]),'user_id']))\n    if len(seeds)!=2:raise ValueError('Required users missing from this snapshot')\n    chosen={(\"Users\",u) for u in seeds};in_window=set()\n    actors={'PullRequests':'author_id','Issues':'author_id','IssueComments':'author_id','PRReviews':'reviewer_id','PRReviewComments':'author_id','Pushes':'user_id'}\n    for name,actor in actors.items():\n        d=ts[name];sel=d[(d[TIME[name]]>=start)&d[actor].isin(seeds)]\n        for i in sel.index:chosen.add((name,int(i)));in_window.add((name,int(i)))\n    # Include FK endpoints recursively; never add unrelated events in shared repos.\n    todo=list(chosen)\n    while todo:\n        table,i=todo.pop();row=ts[table].loc[i]\n        for c,target in FKS[table].items():\n            if c not in row or pd.isna(row[c]):continue\n            j=int(row[c]);key=(target,j)\n            if j in ts[target].index and key not in chosen:\n                chosen.add(key);todo.append(key)\n    h=Digraph('entities',format='svg');h.attr(rankdir='TB',fontname='Arial',label='2026-09-15 00:00 to 09-17 00:00 UTC\\nColored activity rows are in the window; gray nodes are necessary FK context.',labelloc='t');h.attr('node',shape='box',fontname='Arial',fontsize='10')\n    for table,i in sorted(chosen):\n        row=ts[table].loc[i];label=f'{table} {i}'\n        if table=='Users':label+=f'\\nGitHub ID {int(row.github_user_id)}'\n        key=f'{table}__{i}';fill='#b5d8ed' if (table,i) in in_window else '#eeeeee'\n        if table=='Users' and i in seeds:fill='#f5d992'\n        h.node(key,label,style='filled',fillcolor=fill)\n    edges=0\n    for table,i in sorted(chosen):\n        row=ts[table].loc[i]\n        for c,target in FKS[table].items():\n            if c not in row or pd.isna(row[c]):continue\n            j=int(row[c])\n            if (target,j) in chosen:h.edge(f'{table}__{i}',f'{target}__{j}',label=c,fontsize='8');edges+=1\n    h.save(str(out/'entity_graph.dot'))\n    try:h.render(str(out/'entity_graph'),cleanup=True)\n    except Exception as e:print('DOT saved; SVG needs Graphviz:',e)\n    (out/'graph_manifest.json').write_text(json.dumps({'schema_tables':8,'schema_fk_edges':sum(map(len,FKS.values())),'entity_nodes':len(chosen),'entity_edges':edges,'activity_rows_in_window':len(in_window),'start_utc':str(start),'end_utc_exclusive':str(end),'seed_github_ids':[22996444,28983449],'context_policy':'necessary foreign-key endpoints may predate the selected activity window; no future rows or unrelated repo events'},indent=2))\n    print('Graph manifest:',(out/'graph_manifest.json').read_text())\n\nif __name__=='__main__':\n import argparse\n p=argparse.ArgumentParser();p.add_argument('--data',required=True);p.add_argument('--out',default='artifacts/graphs');a=p.parse_args();main(a.data,a.out)\n"}''')
for name, contents in SOURCE_FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents, encoding='utf-8')
print('Project:', PROJECT)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)
print('Dependencies ready')


In [ ]:
DATA = BASE / 'assignment2_data'
# Never delete or replace your project while updating instructor data.
# A new session downloads the current release. Same-session reruns reuse it.
if not DATA.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/nycubigdata-2026fall/BigData2026-Assignment2.git', str(DATA)], check=True)
if not (DATA / 'data/db/Users.parquet').exists():
    raise RuntimeError('Data folder exists but is incomplete; use a NEW empty DATA path and rerun this cell.')
print('Official data:', DATA)
print('Snapshot commit:', subprocess.check_output(['git', '-C', str(DATA), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
OUT = PROJECT / 'artifacts'
subprocess.run([sys.executable, '-m', 'asg2.run', 'benchmark', '--data', str(DATA), '--out', str(OUT), '--trees', '160'], cwd=PROJECT, check=True)


In [ ]:
import pandas as pd
metrics = pd.read_csv(OUT / 'metrics.csv')
summary = metrics.loc[metrics['split'].eq('holdout'), ['model','macro_f1','micro_f1','score']].copy()
summary[['macro_f1','micro_f1','score']] *= 100
print('Historical holdout scores (%) — NOT official November scores')
display(summary)
print((OUT / 'protocol.json').read_text())


In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'scripts/checks.py'), '--data', str(DATA), '--out', str(OUT)], cwd=PROJECT, check=True)


In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'scripts/graphs.py'), '--data', str(DATA), '--out', str(OUT / 'graphs')], cwd=PROJECT, check=True)
# If native Graphviz is absent, DOT files still contain the complete computed graphs.
from IPython.display import SVG, display
for name in ['schema', 'entity_graph']:
    path = OUT / 'graphs' / f'{name}.svg'
    if path.exists(): display(SVG(filename=str(path)))
    else: print('Computed graph saved as DOT:', path.with_suffix('.dot'))


In [ ]:
import shutil
# Save results without large intermediate caches. Keep model pickles locally for final refit.
export = BASE / 'Asg2_Step1_Results'
export.mkdir(exist_ok=True)
for path in OUT.rglob('*'):
    if path.is_file() and 'cache' not in path.parts and path.suffix != '.pkl':
        dest = export / path.relative_to(OUT)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, dest)
archive = shutil.make_archive(str(BASE / 'Asg2_Step1_Results'), 'zip', export)
print('Download:', archive)
print('Baseline and checks finished. Official prediction, final slides and personal recording are still pending.')


## Later final prediction — do not run with October data

After the final Nov11 data update and final model validation, use the CLI `predict` command in README. It expects official `test_input.csv`, refuses an early snapshot, refits the selected model, and runs the teacher's format checker. It has not been executed on future November data today.

The project currently contains relational feature aggregation plus LightGBM, not a graph neural network. A graph-model comparison and fresh forward validation are the next development steps. Do not reuse the October holdout to choose repeated new model settings.
